In [1]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
legal_masks = torch.load("legal_masks.pt")
target = pd.read_csv("target.csv")

In [2]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [3]:
row = states.shape[0]

print(states.shape)
print(legal_masks.shape)

torch.Size([707542, 30, 8, 8])
torch.Size([707542, 4672])


In [4]:
import sys
sys.path.append('..')

In [5]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [6]:
# Train test split
TRAIN_SIZE = int(0.9 * len(states))
assert len(states) == len(policy) == len(value) == len(legal_masks) 

train_states, test_states = states[:TRAIN_SIZE],      states[TRAIN_SIZE:]
train_policy, test_policy = policy[:TRAIN_SIZE],       policy[TRAIN_SIZE:]
train_value,  test_value  = value[:TRAIN_SIZE],        value[TRAIN_SIZE:]
train_masks,  test_masks  = legal_masks[:TRAIN_SIZE],  legal_masks[TRAIN_SIZE:]

assert len(train_states) == len(train_policy) == len(train_value) == len(train_masks)
assert len(test_states)  == len(test_policy)  == len(test_value)  == len(test_masks)

In [7]:

from core import factory

network = factory.build_network("chess")

In [8]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=3e-4, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [9]:
import time
import torch
import numpy as np
from torch import optim
from torch.optim import lr_scheduler
from torch.utils.data import TensorDataset, DataLoader
import itertools

from core.network import PolicyValueNetwork


def evaluate(network, test_states, test_policy, test_value, test_masks,
             policy_loss_fn, value_loss_fn, batch_size=256, eval_samples=4096):
    network.eval()
    total_policy_loss, total_value_loss, n_batches = 0.0, 0.0, 0

    n = min(len(test_states), eval_samples)

    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch_states = test_states[i:i+batch_size]
            batch_policy = test_policy[i:i+batch_size]
            batch_value  = test_value[i:i+batch_size]
            batch_mask   = test_masks[i:i+batch_size]

            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_mask.to(policy_head.device), float("-inf"))

            total_policy_loss += policy_loss_fn(policy_head, batch_policy).item()
            total_value_loss  += value_loss_fn(value_head, batch_value).item()
            n_batches += 1

    network.train()

    return total_policy_loss / n_batches, total_value_loss / n_batches


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          train_states: torch.Tensor,
          train_policy: torch.Tensor,
          train_value: torch.Tensor,
          train_masks: torch.Tensor, 
          policy_loss_fn,
          value_loss_fn,
          test_states: torch.Tensor | None = None,
          test_policy: torch.Tensor | None = None,
          test_value: torch.Tensor | None = None,
          test_masks: torch.Tensor | None = None,
          batch_size: int = 256,
          num_epoch: int | None = None,
          duration_hour: float | None = None,
          save_path: str | None = "pretrain/default.pt"):

    if num_epoch is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_epoch or duration_hour")

    start = time.time()
    step = 0
    best_val_loss = float('inf') # Initialize best validation loss tracking

    # Dataset
    dataset = TensorDataset(train_states, train_policy, train_value.unsqueeze(-1), train_masks)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    total_steps = len(dataloader) * num_epoch if num_epoch is not None else None

    warmup_steps = 200
    warmup = lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps if total_steps else 9999)
    scheduler = lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    has_test = test_states is not None and test_policy is not None and test_value is not None
    if has_test:
        test_states = test_states.to(device=DEVICE)
        test_policy = test_policy.to(device=DEVICE)
        test_value  = test_value.unsqueeze(-1).to(device=DEVICE)

    epoch_iter = range(num_epoch) if num_epoch is not None else itertools.count()

    for _ in epoch_iter:
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        for batch_states, batch_policy, batch_value, batch_masks in dataloader:
            batch_states = batch_states.to(DEVICE)
            batch_policy = batch_policy.to(DEVICE)
            batch_value  = batch_value.to(DEVICE)
            batch_masks  = batch_masks.to(DEVICE)

            optimizer.zero_grad()
            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_masks, float("-inf"))

            policy_loss = policy_loss_fn(policy_head, batch_policy)
            value_loss  = value_loss_fn(value_head, batch_value)
            loss = policy_loss + value_loss
            loss.backward()

            optimizer.step()
            scheduler.step()

            if step % 10 == 0:
                elapsed = time.time() - start
                print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={(value_loss.item()):.4f} | lr: {scheduler.get_last_lr()[0]:.8f} | {elapsed:.0f}s")

            step += 1

        if has_test:
            val_policy_loss, val_value_loss = evaluate(
                network, test_states, test_policy, test_value, test_masks,
                policy_loss_fn, value_loss_fn
            )
            
            current_val_loss = val_policy_loss + val_value_loss
            
            print(f"    [eval @ {step}] val_policy={val_policy_loss:.4f} | val_value={val_value_loss:.4f} | total_val={current_val_loss:.4f}")

            # Check if this is the best model we've seen so far
            if save_path is not None and current_val_loss < best_val_loss:
                print(f"    [save] Validation loss improved from {best_val_loss:.4f} to {current_val_loss:.4f}. Saving model to '{save_path}'...")
                best_val_loss = current_val_loss
                network.save(path=save_path)


In [11]:
train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    train_states=train_states,
    train_policy=train_policy,
    train_value=train_value,
    train_masks=train_masks,
    test_states=test_states,
    test_policy=test_policy,
    test_value=test_value,
    test_masks=test_masks,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=64,
)

[0] loss=5.3453 | policy=4.7215 | value=0.6238 | lr: 0.00000448 | 1s
[10] loss=5.1882 | policy=4.3939 | value=0.7944 | lr: 0.00001934 | 7s
[20] loss=4.8113 | policy=4.0915 | value=0.7198 | lr: 0.00003419 | 13s
[30] loss=4.6032 | policy=4.0572 | value=0.5460 | lr: 0.00004903 | 19s
[40] loss=4.7839 | policy=4.1371 | value=0.6468 | lr: 0.00006389 | 24s
[50] loss=4.4768 | policy=3.8388 | value=0.6380 | lr: 0.00007874 | 30s
[60] loss=4.4538 | policy=3.7562 | value=0.6975 | lr: 0.00009358 | 35s
[70] loss=4.0917 | policy=3.5126 | value=0.5791 | lr: 0.00010844 | 40s
[80] loss=4.4142 | policy=3.7212 | value=0.6930 | lr: 0.00012329 | 47s
[90] loss=3.8803 | policy=3.3878 | value=0.4925 | lr: 0.00013814 | 53s
[100] loss=4.0870 | policy=3.4301 | value=0.6569 | lr: 0.00015299 | 58s
[110] loss=4.0875 | policy=3.1894 | value=0.8981 | lr: 0.00016784 | 63s
[120] loss=4.2864 | policy=3.7068 | value=0.5796 | lr: 0.00018269 | 69s
[130] loss=3.9204 | policy=3.4004 | value=0.5200 | lr: 0.00019754 | 74s
[140]

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:243: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[200] loss=3.9640 | policy=3.3970 | value=0.5670 | lr: 0.00030000 | 112s
[210] loss=4.0675 | policy=3.3880 | value=0.6794 | lr: 0.00030000 | 117s
[220] loss=3.7710 | policy=3.1579 | value=0.6132 | lr: 0.00030000 | 123s
[230] loss=3.7415 | policy=3.2619 | value=0.4796 | lr: 0.00029999 | 128s
[240] loss=3.8556 | policy=3.2972 | value=0.5584 | lr: 0.00029999 | 133s
[250] loss=3.7799 | policy=3.2679 | value=0.5121 | lr: 0.00029998 | 139s
[260] loss=3.6654 | policy=3.1703 | value=0.4951 | lr: 0.00029997 | 144s
[270] loss=3.8487 | policy=3.3640 | value=0.4846 | lr: 0.00029996 | 150s
[280] loss=3.7410 | policy=3.2528 | value=0.4882 | lr: 0.00029995 | 155s
[290] loss=3.7203 | policy=3.2429 | value=0.4774 | lr: 0.00029994 | 161s
[300] loss=3.8376 | policy=3.2182 | value=0.6194 | lr: 0.00029992 | 166s
[310] loss=3.7753 | policy=3.2025 | value=0.5728 | lr: 0.00029991 | 171s
[320] loss=3.8555 | policy=3.2127 | value=0.6429 | lr: 0.00029989 | 177s
[330] loss=3.7716 | policy=3.0894 | value=0.6822 | 

TypeError: PolicyValueNetwork.save() missing 3 required positional arguments: 'game', 'version', and 'file_name'

In [ ]:
import numpy as np
avg_legal = test_masks[:4096].sum(dim=1).float().mean().item()
print("uniform-over-legal baseline:", np.log(avg_legal))

uniform-over-legal baseline: 3.4216014033499937
